# Unit 7: 实战 — 字符级 LSTM 文本生成

## 项目目标

使用字符级 LSTM 语言模型，学习一个文本语料的统计规律，然后**生成风格类似的新文本**。

### 技术栈
- **模型**: 多层 LSTM + Embedding
- **数据**: 莎士比亚作品集 (Tiny Shakespeare)
- **采样策略**: Greedy / Temperature / Top-K
- **指标**: Perplexity (困惑度)

### 完整流水线
```
语料加载 → 字符级Tokenize → 构建词表 → 滑动窗口数据集
    → LSTM模型构建 → 训练循环 → 采样生成 → 结果分析
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import urllib.request
import os

torch.manual_seed(42)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## 7.1 数据加载

使用 Karpathy 的 Tiny Shakespeare 数据集（约 1MB，包含莎士比亚全部作品）。
如果下载失败，使用内置的样本数据。

In [ ]:
def load_shakespeare():
    """下载 Tiny Shakespeare 数据集，失败则返回样本数据"""
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    cache_path = "/tmp/tinyshakespeare.txt"

    if os.path.exists(cache_path):
        with open(cache_path, 'r') as f:
            return f.read()

    try:
        print("正在下载 Tiny Shakespeare 数据集...")
        urllib.request.urlretrieve(url, cache_path)
        with open(cache_path, 'r') as f:
            text = f.read()
        print(f"下载成功! 文本长度: {len(text):,} 字符")
        return text
    except Exception as e:
        print(f"下载失败 ({e})，使用内置样本数据")
        return SAMPLE_TEXT

SAMPLE_TEXT = """
ROMEO:
But, soft! what light through yonder window breaks?
It is the east, and Juliet is the sun.
Arise, fair sun, and kill the envious moon,
Who is already sick and pale with grief,
That thou her maid art far more fair than she:
Be not her maid, since she is envious;
Her vestal livery is but sick and green
And none but fools do wear it; cast it off.

JULIET:
O Romeo, Romeo! wherefore art thou Romeo?
Deny thy father and refuse thy name;
Or, if thou wilt not, be but sworn my love,
And I'll no longer be a Capulet.

HAMLET:
To be, or not to be: that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles,
And by opposing end them? To die: to sleep;
No more; and by a sleep to say we end
The heart-ache and the thousand natural shocks
That flesh is heir to, 'tis a consummation
Devoutly to be wish'd. To die, to sleep;
To sleep: perchance to dream: ay, there's the rub;
For in that sleep of death what dreams may come
When we have shuffled off this mortal coil,
Must give us pause.
"""

text = load_shakespeare()
print(f"前 200 个字符:\n{text[:200]}")

## 7.2 字符级词表构建

统计所有出现的字符，建立 char ↔ id 的双向映射。

In [ ]:
# 构建字符词表
chars = sorted(list(set(text)))
vocab_size = len(chars)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"唯一字符数: {vocab_size}")
print(f"字符列表: {repr(''.join(chars))}")

# 将整个文本编码为整数序列
encoded_text = np.array([char_to_idx[ch] for ch in text], dtype=np.int64)
print(f"编码后序列长度: {len(encoded_text):,}")

## 7.3 滑动窗口数据集

对于字符级语言模型，每个样本是 `seq_length` 个字符，目标是下一个字符。

In [ ]:
class CharDataset(Dataset):
    """字符级语言模型数据集"""
    def __init__(self, encoded_text, seq_length):
        self.data = encoded_text
        self.seq_length = seq_length

    def __len__(self):
        return len(self.data) - self.seq_length

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.seq_length]
        y = self.data[idx + 1:idx + self.seq_length + 1]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

# 配置
seq_length = 100        # 每个训练样本的输入长度
batch_size = 64
train_ratio = 0.9

# 划分训练/验证集
split_idx = int(len(encoded_text) * train_ratio)
train_data = encoded_text[:split_idx]
val_data = encoded_text[split_idx:]

train_dataset = CharDataset(train_data, seq_length)
val_dataset = CharDataset(val_data, seq_length)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

print(f"训练样本数: {len(train_dataset):,}")
print(f"验证样本数: {len(val_dataset):,}")
print(f"每 epoch 训练 batch 数: {len(train_loader)}")

# 查看一个样本
x_sample, y_sample = train_dataset[0]
print(f"\n输入 (前30字符): {repr(''.join(idx_to_char[i.item()] for i in x_sample[:30]))}")
print(f"目标 (前30字符): {repr(''.join(idx_to_char[i.item()] for i in y_sample[:30]))}")
print(f"输入=目标右移一位: {torch.equal(x_sample[1:], y_sample[:-1])}")

## 7.4 LSTM 语言模型

$$
P(w_1, w_2, ..., w_T) = \prod_{t=1}^{T} P(w_t \mid w_1, ..., w_{t-1})
$$

模型架构:
```
Input (token ids) → Embedding → LSTM (多层) → Linear → Logits (vocab_size)
```

In [ ]:
class CharLSTM(nn.Module):
    """字符级 LSTM 语言模型"""
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers=3, dropout=0.3):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)

        # 参数初始化
        self._init_weights()

    def _init_weights(self):
        for name, param in self.named_parameters():
            if 'weight_ih' in name:  # LSTM 输入权重
                nn.init.xavier_uniform_(param)
            elif 'weight_hh' in name:  # LSTM 递归权重
                nn.init.orthogonal_(param)
            elif 'weight' in name:
                nn.init.xavier_uniform_(param)
            elif 'lstm' in name and 'bias' in name:
                nn.init.zeros_(param)
                n = param.size(0)
                start, end = n // 4, n // 2
                param.data[start:end].fill_(1.0)
            elif 'bias' in name:
                nn.init.zeros_(param)

    def forward(self, x, hidden=None):
        """
        Args:
            x: (batch, seq_len) token ids
            hidden: 可选的初始隐藏状态
        Returns:
            logits: (batch, seq_len, vocab_size)
            hidden: 最终隐藏状态 (用于 stateful inference)
        """
        emb = self.embedding(x)                    # (B, T, E)
        lstm_out, hidden = self.lstm(emb, hidden)  # (B, T, H)
        lstm_out = self.dropout(lstm_out)
        logits = self.fc(lstm_out)                 # (B, T, V)
        return logits, hidden

    def init_hidden(self, batch_size):
        """初始化零隐藏状态"""
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)
        return (h0, c0)

# 模型超参数
embed_dim = 128
hidden_size = 512
num_layers = 3
dropout = 0.3

model = CharLSTM(vocab_size, embed_dim, hidden_size, num_layers, dropout).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"模型参数: {total_params:,} (可训练: {trainable_params:,})")
print(f"词表大小: {vocab_size}")

# 测试前向传播
x_test = torch.randint(0, vocab_size, (4, seq_length)).to(device)
logits, hidden = model(x_test)
print(f"输入 shape: {x_test.shape}")
print(f"输出 shape: {logits.shape}")

## 7.5 训练循环

使用 CrossEntropyLoss，计算 Perplexity = $e^{\text{loss}}$ 作为评价指标。

In [ ]:
def train_epoch(model, loader, optimizer, criterion, clip_grad=1.0):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits, _ = model(x)
        loss = criterion(logits.view(-1, vocab_size), y.view(-1))
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits, _ = model(x)
            loss = criterion(logits.view(-1, vocab_size), y.view(-1))
            total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
# 训练
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.002, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.8)

num_epochs = 25
train_losses = []
val_losses = []
train_ppls = []
val_ppls = []

print(f"开始训练 {num_epochs} 个 epoch...\n")

for epoch in range(1, num_epochs + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss = evaluate(model, val_loader, criterion)

    scheduler.step()

    train_ppl = np.exp(train_loss)
    val_ppl = np.exp(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_ppls.append(train_ppl)
    val_ppls.append(val_ppl)

    if epoch % 5 == 0 or epoch == 1:
        lr = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch:2d}/{num_epochs} | lr={lr:.4f} | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Val PPL: {val_ppl:.2f}")

print(f"\n训练完成! 最终 Val Perplexity: {val_ppls[-1]:.2f}")

In [ ]:
# 可视化训练曲线
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label='Train Loss', linewidth=1.5)
ax1.plot(val_losses, label='Val Loss', linewidth=1.5)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(train_ppls, label='Train PPL', linewidth=1.5)
ax2.plot(val_ppls, label='Val PPL', linewidth=1.5)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Perplexity')
ax2.set_title('Training & Validation Perplexity')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Perplexity = exp(loss)，越低越好")
print(f"随机猜测的 PPL = {vocab_size}（均匀分布）")

## 7.6 文本生成 — 三种采样策略

生成新文本的关键在于**采样策略**：如何从模型输出的概率分布中选择下一个字符。

| 策略 | 原理 | 特点 |
|------|------|------|
| **Greedy** | 选择概率最高的字符 | 确定性，重复度高 |
| **Temperature** | 先缩放 logits 再 softmax | 控制随机性，T↑=更随机，T↓=更确定 |
| **Top-K** | 只从概率前 K 的 token 中采样 | 避免低概率噪声，保留多样性 |

In [ ]:
def sample_greedy(logits):
    """贪心采样: 选择概率最高的 token"""
    probs = F.softmax(logits, dim=-1)
    return torch.argmax(probs, dim=-1)


def sample_temperature(logits, temperature=0.8):
    """
    Temperature 采样:
    - T → 0: 趋近贪心
    - T = 1: 原始分布
    - T → ∞: 趋近均匀分布
    """
    logits = logits / max(temperature, 1e-9)
    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze(-1)


def sample_top_k(logits, k=10, temperature=0.8):
    """
    Top-K 采样:
    只保留概率最高的 K 个 token，将其它置为 -inf，然后做 temperature 采样
    """
    logits = logits / max(temperature, 1e-9)
    top_k_values, _ = torch.topk(logits, k, dim=-1)
    min_top_k = top_k_values[:, -1].unsqueeze(-1)
    logits = torch.where(logits < min_top_k, torch.tensor(float('-inf'), device=device), logits)
    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze(-1)

In [ ]:
def generate(model, seed_text, char_to_idx, idx_to_char, length=200,
             strategy='temperature', temperature=0.8, top_k=10):
    """
    根据 seed 文本生成新文本

    Args:
        model: 训练好的 CharLSTM 模型
        seed_text: 种子文本
        length: 生成的总字符数
        strategy: 'greedy', 'temperature', 'top_k'
        temperature: Temperature 参数 (greedy 忽略)
        top_k: Top-K 参数 (仅 top_k 策略)
    Returns:
        生成的完整文本 (seed + 生成部分)
    """
    model.eval()

    # 选择采样函数
    if strategy == 'greedy':
        sample_fn = lambda logits: sample_greedy(logits)
    elif strategy == 'top_k':
        sample_fn = lambda logits: sample_top_k(logits, k=top_k, temperature=temperature)
    else:
        sample_fn = lambda logits: sample_temperature(logits, temperature=temperature)

    # 编码 seed
    input_ids = torch.tensor(
        [[char_to_idx[ch] for ch in seed_text if ch in char_to_idx]],
        dtype=torch.long, device=device
    )

    generated_chars = list(seed_text)
    hidden = None

    with torch.no_grad():
        # 先用 seed 文本初始化隐藏状态
        if input_ids.size(1) > 1:
            logits, hidden = model(input_ids[:, :-1])
            last_input = input_ids[:, -1:]
        else:
            last_input = input_ids

        # 逐字符生成
        for _ in range(length):
            logits, hidden = model(last_input, hidden)
            next_token = sample_fn(logits[:, -1, :])

            if next_token.item() not in idx_to_char:
                break

            generated_chars.append(idx_to_char[next_token.item()])
            last_input = next_token.unsqueeze(1)

    return ''.join(generated_chars)

In [ ]:
# 准备几个 seed 文本
seeds = [
    "ROMEO:\n",
    "JULIET:\n",
    "To be, or not to be",
    "The king ",
]

seed = seeds[0]
print(f"Seed: {repr(seed)}")

In [ ]:
# 策略1: Greedy
print("=" * 60)
print("【Greedy 采样】—— 总是选概率最高的，可能产生重复")
print("=" * 60)

generated_greedy = generate(
    model, seed, char_to_idx, idx_to_char,
    length=300, strategy='greedy'
)
print(generated_greedy)

In [ ]:
# 策略2: Temperature 采样 (不同 temperature)
for temp in [0.3, 0.7, 1.0, 1.5]:
    print("=" * 60)
    print(f"【Temperature = {temp}】")
    if temp < 0.5:
        print("低温: 接近贪心，输出较确定")
    elif temp < 1.0:
        print("中温: 平衡确定性与随机性")
    elif temp == 1.0:
        print("原始分布: 不做缩放")
    else:
        print("高温: 分布更均匀，输出更随机")
    print("=" * 60)

    generated = generate(
        model, seed, char_to_idx, idx_to_char,
        length=200, strategy='temperature', temperature=temp
    )
    print(generated)
    print()

In [ ]:
# 策略3: Top-K 采样
for k_val in [3, 10, 50]:
    print("=" * 60)
    print(f"【Top-K = {k_val}, Temperature = 0.8】")
    if k_val <= 5:
        print("小 K: 输出较保守，但比贪心有多样性")
    elif k_val <= 20:
        print("中 K: 推荐设置，平衡质量与多样性")
    else:
        print("大 K: 较随机，可能有语法错误")
    print("=" * 60)

    generated = generate(
        model, seed, char_to_idx, idx_to_char,
        length=200, strategy='top_k', temperature=0.8, top_k=k_val
    )
    print(generated)
    print()

## 7.7 不同 Seed 的生成效果

同一个模型，不同的 seed 文本会引导出不同风格的生成内容。

In [ ]:
for seed in seeds:
    print("=" * 60)
    print(f"Seed: {repr(seed)}")
    print("=" * 60)

    generated = generate(
        model, seed, char_to_idx, idx_to_char,
        length=200, strategy='top_k', temperature=0.7, top_k=15
    )
    print(generated)
    print()

## 7.8 模型保存与加载

保存训练好的模型，以便后续使用。

In [ ]:
save_path = "/tmp/char_lstm_shakespeare.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'char_to_idx': char_to_idx,
    'idx_to_char': idx_to_char,
    'vocab_size': vocab_size,
    'embed_dim': embed_dim,
    'hidden_size': hidden_size,
    'num_layers': num_layers,
}, save_path)

print(f"模型已保存到 {save_path}")

# 加载模型
checkpoint = torch.load(save_path, map_location=device, weights_only=True)
loaded_model = CharLSTM(
    checkpoint['vocab_size'], checkpoint['embed_dim'],
    checkpoint['hidden_size'], checkpoint['num_layers']
).to(device)
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model.eval()

print("模型已加载，验证一致性:")
with torch.no_grad():
    x = torch.randint(0, vocab_size, (1, 10)).to(device)
    out_orig, _ = model(x)
    out_loaded, _ = loaded_model(x)
    diff = (out_orig - out_loaded).abs().max().item()
    print(f"  输出最大差异: {diff:.10f}")

## 项目总结

### 你学到了什么

| 阶段 | 技能 |
|------|------|
| 数据处理 | 字符级 tokenization、词表构建、滑动窗口 |
| 模型设计 | Embedding + LSTM 堆叠 + 参数初始化 |
| 训练 | BPTT、梯度裁剪、学习率调度、Perplexity 评估 |
| 推理 | 逐字符生成、hidden state 跨步传递 |
| 采样 | Greedy / Temperature / Top-K 策略对比 |

### 改进方向
- 增加训练数据量或训练更多的 epoch
- 尝试更深/更宽的 LSTM 配置
- 使用 Transformer / GPT 架构
- 实现 Beam Search 解码
- 加入 Attention 机制

### 练习
1. 修改模型为词级 LSTM，对比生成效果
2. 实现 Top-P (Nucleus) 采样策略
3. 用自己的文本数据（如歌词、小说）训练个性化生成器